In [ ]:
import os
import gc
import json
import torch
import pandas as pd

from PIL import Image
from tqdm import tqdm
from google.colab import drive

drive.mount("/content/drive")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
QUESTIONS = {
    "scene_description": (
        "Briefly describe the visible body region and the general scene. "
        "Do not invent details that are not visible."
    ),

    "has_tattoo": (
        "Is there any visible tattoo in the image? "
        "Answer only: yes, no, or uncertain."
    ),

    "tattoo_count": (
        "How many visible tattoos are there? "
        "Answer with a number if possible, otherwise say uncertain."
    ),

    "tattoo_location": (
        "If there is a visible tattoo, where is it located on the body? "
        "Choose the closest option: arm, forearm, hand, chest, abdomen, back, shoulder, neck, leg, foot, multiple locations, other, uncertain."
    ),

    "tattoo_size": (
        "What is the approximate visible size of the tattoo relative to the visible body area? "
        "Choose one: small, medium, large, uncertain."
    ),

    "tattoo_color": (
        "Does the tattoo contain visible colors, or does it appear black/gray only? "
        "Answer only: colored, black/gray, uncertain."
    ),

    "tattoo_style": (
        "What is the visual style of the tattoo? "
        "Choose one: line-art, filled/solid, shaded, realistic, text-only, simple symbol, complex illustration, uncertain."
    ),

    "tattoo_category": (
        "What is the main type of tattoo motif? "
        "Choose one: animal, person/face, text/lettering, symbol, object, plant/flower, geometric, abstract, fantasy/creature, other, uncertain."
    ),

    "tattoo_motif_description": (
        "Briefly describe the visible tattoo motif. "
        "If unclear, answer uncertain. Do not guess."
    ),

    "contains_text": (
        "Does the tattoo contain readable text or lettering? "
        "Answer only: yes, no, or uncertain."
    ),

    "text_content": (
        "If readable text is visible in the tattoo, transcribe it exactly. "
        "If none or unclear, answer none or uncertain."
    ),

    "visual_complexity": (
        "How visually complex is the tattoo? "
        "Choose one: simple, moderate, complex, uncertain."
    ),

    "confidence": (
        "How confident are you in your tattoo assessment? "
        "Choose one: low, medium, high."
    )
}

Análisis estadístico para 5 características sencillas de etiquetar. Trabajaremos con el conjunto test.

In [ ]:
import os
import pandas as pd
from PIL import Image
from IPython.display import display, clear_output

BASE = "/content/drive/MyDrive/Tattoo-YOLO"
SPLIT = "test"

IMAGE_DIR = os.path.join(BASE, SPLIT, "images")
MANUAL_LABELS_OUT = os.path.join(BASE, SPLIT, "manual_labels_test.csv")

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

print("Image dir:", IMAGE_DIR)
print("Output CSV:", MANUAL_LABELS_OUT)

In [ ]:
def list_images(image_dir):
    return sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(IMG_EXTENSIONS)
    ])

def load_existing_labels(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame()

def save_labels(df, path):
    df.to_csv(path, index=False)
    print("Guardado en:", path)

def show_image(image_path):
    img = Image.open(image_path).convert("RGB")
    display(img)

def ask_choice(field, options, description=None):
    print(f"\n{field}")
    if description:
        print(description)

    for i, opt in enumerate(options):
        print(f"{i}: {opt}")

    while True:
        value = input("Elige número o escribe la opción: ").strip().lower()

        if value.isdigit():
            idx = int(value)
            if 0 <= idx < len(options):
                return options[idx]

        if value in options:
            return value

        print("Valor no válido. Intenta otra vez.")

In [ ]:
CHOICES = {
    "has_tattoo": ["yes", "no"],
    "tattoo_size": ["small", "medium", "large"],
    "tattoo_color": ["black_and_gray", "colored"],
    "contains_text": ["yes", "no"],
    "tattoo_count": ["single", "multiple"]
}

DESCRIPTIONS = {
    "has_tattoo": (
        "Is there any visible tattoo in the image?"
    ),

    "tattoo_size": (
        "Estimated real-world tattoo size, considering both body coverage and visual density.\n\n"

        "- small: approximately fist-sized or smaller, OR slightly larger but made mostly of thin lines with little fill/shading; "
        "small isolated symbols, fine-line tattoos, short text, or minimal designs.\n\n"

        "- medium: clearly larger than a small tattoo and visually more substantial, but not covering a major body area; "
        "moderate forearm/chest/leg pieces or designs with noticeable fill, shading, or complexity.\n\n"

        "- large: covers a major body area such as most of a forearm, a large chest/back/leg section, half an arm or more, "
        "sleeve-like tattoos, or very visually dense/heavily filled compositions.\n\n"

        "Important: estimate the real tattoo size relative to the body part, not relative to the image crop. "
        "Also consider line thickness, fill amount, shading, and visual density. "
        "A larger but very thin fine-line tattoo can still be considered small."
    ),

    "tattoo_color": (
        "Tattoo color style:\n"
        "- black_and_gray: black, gray, or grayscale tattoo only\n"
        "- colored: clearly contains visible colors such as red, blue, green, yellow, etc. It has to be clear not a minimal part"
    ),

    "contains_text": (
        "Does the tattoo contain readable text using the Latin alphabet "
        "(standard Western letters and words)?\n"
        "Do not count Chinese characters, Japanese symbols, runes, abstract glyphs, or decorative symbols as text."
    ),

    "tattoo_count": (
        "Number of visible tattoos:\n"
        "- single: one tattoo or one continuous tattoo with relation composition\n"
        "- multiple: several separate tattoos"
    )
}

Primero etiquetas manuales para luego comprobar la tasa de acierto

In [ ]:
image_paths = list_images(IMAGE_DIR)
labels_df = load_existing_labels(MANUAL_LABELS_OUT)

already_done = set(labels_df["filename"]) if not labels_df.empty and "filename" in labels_df.columns else set()

print("Total imágenes en test:", len(image_paths))
print("Ya etiquetadas:", len(already_done))
print("Pendientes:", len(image_paths) - len(already_done))

if not labels_df.empty:
    display(labels_df.tail())

In [ ]:
labels_df = load_existing_labels(MANUAL_LABELS_OUT)
already_done = set(labels_df["filename"]) if not labels_df.empty and "filename" in labels_df.columns else set()

pending = [
    p for p in image_paths
    if os.path.basename(p) not in already_done
]

if not pending:
    print("No quedan imágenes pendientes.")
    CURRENT_IMAGE_PATH = None
else:
    CURRENT_IMAGE_PATH = pending[0]
    CURRENT_FILENAME = os.path.basename(CURRENT_IMAGE_PATH)

    print("Imagen pendiente:")
    print(CURRENT_FILENAME)
    print("\nOpciones:")
    print("has_tattoo: yes / no")
    print("tattoo_size: small / medium / large")
    print("tattoo_color: black_and_gray / colored")
    print("contains_text: yes / no")
    print("tattoo_count: single / multiple")

    display(Image.open(CURRENT_IMAGE_PATH).convert("RGB"))

In [ ]:
# Modificar valores
has_tattoo = "yes"              # yes / no
tattoo_size = "large"          # small / medium / large
tattoo_color = "colored" # black_and_gray / colored
contains_text = "no"            # yes / no
tattoo_count = "multiple"         # single / multiple

# Guardar
VALID_VALUES = {
    "has_tattoo": ["yes", "no"],
    "tattoo_size": ["small", "medium", "large"],
    "tattoo_color": ["black_and_gray", "colored"],
    "contains_text": ["yes", "no"],
    "tattoo_count": ["single", "multiple"]
}

row = {
    "filename": CURRENT_FILENAME,
    "image_path": CURRENT_IMAGE_PATH,
    "has_tattoo": has_tattoo,
    "tattoo_size": tattoo_size,
    "tattoo_color": tattoo_color,
    "contains_text": contains_text,
    "tattoo_count": tattoo_count
}

for key, valid in VALID_VALUES.items():
    if row[key] not in valid:
        raise ValueError(f"Valor inválido en {key}: {row[key]}. Opciones válidas: {valid}")

labels_df = load_existing_labels(MANUAL_LABELS_OUT)

labels_df = labels_df[labels_df["filename"] != CURRENT_FILENAME] if not labels_df.empty else labels_df

labels_df = pd.concat(
    [labels_df, pd.DataFrame([row])],
    ignore_index=True
)

save_labels(labels_df, MANUAL_LABELS_OUT)

print("Etiqueta guardada:")
display(pd.DataFrame([row]))

print(f"Progreso: {len(labels_df)} / {len(image_paths)}")

Eliminar imágenes mal etiquetadas para rehacer

In [ ]:
# Ruta del CSV
MANUAL_LABELS_OUT = "/content/drive/MyDrive/Tattoo-YOLO/test/manual_labels_test.csv"

# Cargar etiquetas
labels_df = pd.read_csv(MANUAL_LABELS_OUT)

print("Total etiquetas actuales:", len(labels_df))

# Imagen a eliminar
filename_to_remove = ""

# Eliminar
before = len(labels_df)

labels_df = labels_df[
    labels_df["filename"] != filename_to_remove
].reset_index(drop=True)

after = len(labels_df)

removed = before - after

if removed > 0:
    labels_df.to_csv(MANUAL_LABELS_OUT, index=False)
    print(f"Etiqueta eliminada correctamente: {filename_to_remove}")
else:
    print("No se encontró ese filename.")

print("Total etiquetas después:", len(labels_df))

Preguntas e instrucciones para las respuestas de los LLM

In [ ]:
BASE = "/content/drive/MyDrive/Tattoo-YOLO"
SPLIT = "test"

IMAGE_DIR = os.path.join(BASE, SPLIT, "images")
MANUAL_LABELS_PATH = os.path.join(BASE, SPLIT, "manual_labels_test.csv")

IMG_EXTENSIONS = (".jpg", ".jpeg", ".png")

In [ ]:
SYSTEM_INSTRUCTION = (
    "You are classifying visible tattoos in images for a research dataset. "
    "Use only the visible tattoo ink. Do not consider skin color, clothes, background, lighting, shadows, or image filters. "
    "Be conservative and do not invent details. "
    "Return exactly one label and nothing else."
)

EVAL_QUESTIONS = {
    "has_tattoo": (
        "Is there visible tattoo ink on the person's skin? "
        "Ignore clothing patterns, shadows, body hair, jewelry, and background objects. "
        "Answer exactly one label: yes or no."
    ),

    "tattoo_size": (
        "Classify the real-world size of the largest visible tattoo.\n\n"
        "Judge the tattoo size relative to the body part, not relative to the image crop or zoom.\n"
        "Consider both physical coverage and visual density, including line thickness, fill, shading, and detail.\n\n"

        "small = small isolated tattoo, approximately fist-sized or smaller; short word; small symbol; minimal fine-line design; or lightly filled design with limited body coverage.\n\n"

        "medium = tattoo clearly larger than small but not covering an entire major body region. "
        "It occupies a noticeable and substantial part of a body area, but still leaves a large part of that body area untattooed. "
        "Examples include a moderate forearm piece, a moderate chest/pectoral piece, a moderate calf/leg piece, or a design covering part but not most/all of the visible limb or torso region.\n\n"

        "large = tattoo covering most or all of a major visible body region, sleeve-like coverage, very extensive chest/back/leg coverage, or a very dense/heavily filled tattoo spanning a large area.\n\n"

        "Do not classify as large only because the photo is close-up or tightly cropped. "
        "Do not classify as small if the tattoo covers a substantial body area, even if it is not a full sleeve or full-region tattoo.\n\n"

        "Answer exactly one label: small, medium, or large."
    ),

    "tattoo_color": (
        "Classify the color of the tattoo ink only. "
        "black_and_gray = tattoo ink appears black, gray, grayscale, or dark monochrome only. "
        "colored = tattoo ink clearly contains non-gray ink such as red, blue, green, yellow, purple, orange, etc. "
        "Do NOT count skin tone, redness, lighting, shadows, background, clothing, or tiny ambiguous traces as color. "
        "If unsure, choose black_and_gray. "
        "Answer exactly one label: black_and_gray or colored."
    ),

    "contains_text": (
        "Decide whether any visible tattoo contains Latin alphabet text or numbers.\n\n"
        "yes = visible Latin letters A-Z, words, names, initials, dates, or numbers are present. "
        "The text does not need to be fully readable. It may be rotated, upside down, mirrored, curved, stylized, or partially obscured; if Latin letters or numbers are recognizable as text, answer yes.\n\n"

        "no = the tattoo contains no Latin alphabet text or numbers. "
        "Do NOT count Chinese/Japanese/Korean characters, Arabic script, runes, symbols, logos, decorative glyphs, geometric patterns, abstract marks, or ornamental shapes as text for this task.\n\n"

        "Answer exactly one label: yes or no."
    ),

    "tattoo_count": (
        "Classify whether the image shows one tattoo or multiple separate tattoos.\n\n"
        "single = one tattoo, one continuous tattoo composition, or several connected/related parts forming one overall design.\n\n"
        "multiple = two or more clearly separate tattoos, even if they are on the same body part. "
        "Choose multiple when there are separate designs, separate words/symbols, or tattoo elements divided by visible skin gaps and not clearly part of one continuous composition.\n\n"

        "Do not require the tattoos to be far apart. "
        "Only choose multiple if the separate markings are clearly tattoos and visually distinct from each other.\n\n"

        "Answer exactly one label: single or multiple."
    )
}

In [ ]:
def list_images(image_dir):
    return sorted([
        os.path.join(image_dir, f)
        for f in os.listdir(image_dir)
        if f.lower().endswith(IMG_EXTENSIONS)
    ])

def load_image(path):
    return Image.open(path).convert("RGB")

def clean_answer(answer):
    if answer is None:
        return ""

    ans = str(answer).strip().lower()
    ans = ans.replace(".", "").replace(",", "").replace(":", "")
    ans = ans.replace("\n", " ")

    # Normalización básica
    if "black" in ans or "gray" in ans or "grey" in ans:
        if "colored" not in ans and "colorful" not in ans:
            return "black_and_gray"

    if "colored" in ans or "colour" in ans or "colorful" in ans:
        return "colored"

    if "yes" in ans:
        return "yes"

    if "no" in ans:
        return "no"

    if "small" in ans:
        return "small"

    if "medium" in ans:
        return "medium"

    if "large" in ans:
        return "large"

    if "multiple" in ans or "several" in ans:
        return "multiple"

    if "single" in ans or "one" in ans:
        return "single"

    return ans

def normalize_prediction(field, answer):
    ans = clean_answer(answer)

    valid_values = {
        "has_tattoo": ["yes", "no"],
        "tattoo_size": ["small", "medium", "large"],
        "tattoo_color": ["black_and_gray", "colored"],
        "contains_text": ["yes", "no"],
        "tattoo_count": ["single", "multiple"]
    }

    if ans in valid_values[field]:
        return ans

    return "invalid"

Primer Modelo - Moondream2

In [ ]:
OUT_DIR = os.path.join(BASE, SPLIT, "moondream2_eval")
os.makedirs(OUT_DIR, exist_ok=True)

MOONDREAM_JSON_OUT = os.path.join(OUT_DIR, "moondream2_predictions.json")
MOONDREAM_CSV_OUT = os.path.join(OUT_DIR, "moondream2_predictions.csv")

print("Image dir:", IMAGE_DIR)
print("Manual labels:", MANUAL_LABELS_PATH)

In [ ]:
!apt-get update -qq
!apt-get install -y -qq libvips42

!pip uninstall -y transformers
!pip install -q "transformers==4.51.1" accelerate einops pillow pandas tqdm pyvips

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MOONDREAM_ID = "vikhyatk/moondream2"
MOONDREAM_REVISION = "2025-01-09"

moondream_model = AutoModelForCausalLM.from_pretrained(
    MOONDREAM_ID,
    revision=MOONDREAM_REVISION,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
).to(device)

moondream_tokenizer = AutoTokenizer.from_pretrained(
    MOONDREAM_ID,
    revision=MOONDREAM_REVISION,
    trust_remote_code=True
)

print("Moondream2 cargado correctamente")

In [ ]:
def ask_moondream(image, question):
    full_question = SYSTEM_INSTRUCTION + "\n\n" + question

    with torch.no_grad():
        encoded_image = moondream_model.encode_image(image)
        answer = moondream_model.answer_question(
            encoded_image,
            full_question,
            moondream_tokenizer
        )

    return answer.strip()

def analyze_image_moondream(image_path):
    image = load_image(image_path)

    raw_answers = {}
    normalized_answers = {}

    for field, question in EVAL_QUESTIONS.items():
        try:
            raw = ask_moondream(image, question)
        except Exception as e:
            raw = f"ERROR: {str(e)}"

        raw_answers[field] = raw
        normalized_answers[field] = normalize_prediction(field, raw)

    return {
        "filename": os.path.basename(image_path),
        "image_path": image_path,
        "model": "moondream2",
        "raw_answers": raw_answers,
        "predictions": normalized_answers
    }

In [ ]:
# ============================================
# Procesamiento completo Moondream2 (estable)
# ============================================

import json
import gc
import pandas as pd
from tqdm import tqdm

manual_df = pd.read_csv(MANUAL_LABELS_PATH)
manual_filenames = set(manual_df["filename"].tolist())

all_image_paths = [
    p for p in list_images(IMAGE_DIR)
    if os.path.basename(p) in manual_filenames
]

print("Total imágenes:", len(all_image_paths))

if os.path.exists(MOONDREAM_JSON_OUT):
    with open(MOONDREAM_JSON_OUT, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

processed = set(r["filename"] for r in results)

pending_paths = [
    p for p in all_image_paths
    if os.path.basename(p) not in processed
]

print("Pendientes:", len(pending_paths))


def save_moondream_results(results):

    with open(MOONDREAM_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    rows = []

    for r in results:

        row = {
            "filename": r["filename"],
            "image_path": r["image_path"],
            "model": r["model"]
        }

        for field in EVAL_QUESTIONS.keys():

            row[f"{field}_raw"] = r["raw_answers"].get(field)
            row[field] = r["predictions"].get(field)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(MOONDREAM_CSV_OUT, index=False)


SAVE_EVERY = 10

for idx, path in enumerate(tqdm(pending_paths)):

    try:

        result = analyze_image_moondream(path)
        results.append(result)

    except Exception as e:

        print("\nERROR:", os.path.basename(path))
        print(e)

        results.append({
            "filename": os.path.basename(path),
            "image_path": path,
            "model": "moondream2",
            "error": str(e),
            "raw_answers": {},
            "predictions": {}
        })

    if (idx + 1) % SAVE_EVERY == 0:

        save_moondream_results(results)

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"\nGuardadas {idx + 1} imágenes")

save_moondream_results(results)

print("\nProceso terminado")
print("CSV:", MOONDREAM_CSV_OUT)

Calculamos la precisión del primer modelo

In [ ]:
# ============================================
# Comparar Moondream2 contra etiquetas manuales
# ============================================

manual_df = pd.read_csv(MANUAL_LABELS_PATH)
pred_df = pd.read_csv(MOONDREAM_CSV_OUT)

eval_fields = [
    "has_tattoo",
    "tattoo_size",
    "tattoo_color",
    "contains_text",
    "tattoo_count"
]

merged = manual_df.merge(
    pred_df,
    on="filename",
    suffixes=("_manual", "_moondream")
)

print("Imágenes comparadas:", len(merged))

scores = []

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_moondream"

    correct = merged[manual_col] == merged[pred_col]

    scores.append({
        "model": "moondream2",
        "field": field,
        "correct": int(correct.sum()),
        "total": int(len(correct)),
        "accuracy": float(correct.mean())
    })

scores_df = pd.DataFrame(scores)
display(scores_df)

In [ ]:
# Ver errores por variable

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_moondream"
    raw_col = field + "_raw"

    errors = merged[merged[manual_col] != merged[pred_col]][[
        "filename",
        manual_col,
        pred_col,
        raw_col
    ]]

    print("\n" + "=" * 80)
    print("Errores en:", field)
    print("=" * 80)
    display(errors)

Segundo Modelo - Llava

In [ ]:
!pip install -U bitsandbytes

In [ ]:
import os
os.kill(os.getpid(), 9)

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
import os
import json
import pandas as pd
from tqdm import tqdm

LLAVA_ID = "llava-hf/llava-1.5-7b-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

llava_model = LlavaForConditionalGeneration.from_pretrained(
    LLAVA_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

llava_processor = AutoProcessor.from_pretrained(LLAVA_ID)

print("LLaVA cargado correctamente")

In [ ]:
LLAVA_OUT_DIR = os.path.join(BASE, SPLIT, "llava_eval")
os.makedirs(LLAVA_OUT_DIR, exist_ok=True)

LLAVA_JSON_OUT = os.path.join(LLAVA_OUT_DIR, "llava_predictions.json")
LLAVA_CSV_OUT = os.path.join(LLAVA_OUT_DIR, "llava_predictions.csv")

print("JSON:", LLAVA_JSON_OUT)
print("CSV:", LLAVA_CSV_OUT)

In [ ]:
def ask_llava(image, question):
    full_question = SYSTEM_INSTRUCTION + "\n\n" + question

    prompt = f"USER: <image>\n{full_question}\nASSISTANT:"

    inputs = llava_processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        output = llava_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    decoded = llava_processor.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    if "ASSISTANT:" in decoded:
        decoded = decoded.split("ASSISTANT:")[-1]

    return decoded.strip()


def analyze_image_llava(image_path):
    image = load_image(image_path)

    raw_answers = {}
    normalized_answers = {}

    for field, question in EVAL_QUESTIONS.items():
        try:
            raw = ask_llava(image, question)
        except Exception as e:
            raw = f"ERROR: {str(e)}"

        raw_answers[field] = raw
        normalized_answers[field] = normalize_prediction(field, raw)

    return {
        "filename": os.path.basename(image_path),
        "image_path": image_path,
        "model": "llava_1_5_7b",
        "raw_answers": raw_answers,
        "predictions": normalized_answers
    }

In [ ]:
# ============================================
# Procesamiento completo LLaVA (estable)
# ============================================

import json
import gc
import pandas as pd
from tqdm import tqdm

manual_df = pd.read_csv(MANUAL_LABELS_PATH)
manual_filenames = set(manual_df["filename"].tolist())

all_image_paths = [
    p for p in list_images(IMAGE_DIR)
    if os.path.basename(p) in manual_filenames
]

print("Total imágenes:", len(all_image_paths))

if os.path.exists(LLAVA_JSON_OUT):
    with open(LLAVA_JSON_OUT, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

processed = set(r["filename"] for r in results)

pending_paths = [
    p for p in all_image_paths
    if os.path.basename(p) not in processed
]

print("Pendientes:", len(pending_paths))


def save_llava_results(results):

    with open(LLAVA_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    rows = []

    for r in results:

        row = {
            "filename": r["filename"],
            "image_path": r["image_path"],
            "model": r["model"]
        }

        for field in EVAL_QUESTIONS.keys():

            row[f"{field}_raw"] = r["raw_answers"].get(field)
            row[field] = r["predictions"].get(field)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(LLAVA_CSV_OUT, index=False)


SAVE_EVERY = 10

for idx, path in enumerate(tqdm(pending_paths)):

    try:

        result = analyze_image_llava(path)
        results.append(result)

    except Exception as e:

        print("\nERROR:", os.path.basename(path))
        print(e)

        results.append({
            "filename": os.path.basename(path),
            "image_path": path,
            "model": "llava_1_5_7b",
            "error": str(e),
            "raw_answers": {},
            "predictions": {}
        })

    if (idx + 1) % SAVE_EVERY == 0:

        save_llava_results(results)

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"\nGuardadas {idx + 1} imágenes")

save_llava_results(results)

print("\nProceso terminado")
print("CSV:", LLAVA_CSV_OUT)

In [ ]:
manual_df = pd.read_csv(MANUAL_LABELS_PATH)
pred_df = pd.read_csv(LLAVA_CSV_OUT)

eval_fields = [
    "has_tattoo",
    "tattoo_size",
    "tattoo_color",
    "contains_text",
    "tattoo_count"
]

merged = manual_df.merge(
    pred_df,
    on="filename",
    suffixes=("_manual", "_llava")
)

print("Imágenes comparadas:", len(merged))

scores = []

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_llava"

    correct = merged[manual_col] == merged[pred_col]

    scores.append({
        "model": "llava_1_5_7b",
        "field": field,
        "correct": int(correct.sum()),
        "total": int(len(correct)),
        "accuracy": float(correct.mean())
    })

llava_scores_df = pd.DataFrame(scores)
display(llava_scores_df)

In [ ]:
for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_llava"
    raw_col = field + "_raw"

    errors = merged[merged[manual_col] != merged[pred_col]][[
        "filename",
        manual_col,
        pred_col,
        raw_col
    ]]

    print("\n" + "=" * 80)
    print("Errores en:", field)
    print("=" * 80)
    display(errors)

Tercer Modelo - Qween

In [ ]:
!pip install -q qwen-vl-utils

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

QWEN_ID = "Qwen/Qwen2-VL-2B-Instruct"

qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_ID,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

qwen_processor = AutoProcessor.from_pretrained(QWEN_ID)

print("Qwen2-VL cargado correctamente")

In [ ]:
QWEN_OUT_DIR = os.path.join(BASE, SPLIT, "qwen2vl_eval")
os.makedirs(QWEN_OUT_DIR, exist_ok=True)

QWEN_JSON_OUT = os.path.join(QWEN_OUT_DIR, "qwen2vl_predictions.json")
QWEN_CSV_OUT = os.path.join(QWEN_OUT_DIR, "qwen2vl_predictions.csv")

print("JSON:", QWEN_JSON_OUT)
print("CSV:", QWEN_CSV_OUT)

In [ ]:
def ask_qwen(image_path, question):
    full_question = SYSTEM_INSTRUCTION + "\n\n" + question

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image_path},
                {"type": "text", "text": full_question}
            ]
        }
    ]

    text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    image_inputs, video_inputs = process_vision_info(messages)

    inputs = qwen_processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text.strip()


def analyze_image_qwen(image_path):
    raw_answers = {}
    normalized_answers = {}

    for field, question in EVAL_QUESTIONS.items():
        try:
            raw = ask_qwen(image_path, question)
        except Exception as e:
            raw = f"ERROR: {str(e)}"

        raw_answers[field] = raw
        normalized_answers[field] = normalize_prediction(field, raw)

    return {
        "filename": os.path.basename(image_path),
        "image_path": image_path,
        "model": "qwen2_vl_2b",
        "raw_answers": raw_answers,
        "predictions": normalized_answers
    }

In [ ]:
import json
import gc
import pandas as pd
from tqdm import tqdm

manual_df = pd.read_csv(MANUAL_LABELS_PATH)
manual_filenames = set(manual_df["filename"].tolist())

all_image_paths = [
    p for p in list_images(IMAGE_DIR)
    if os.path.basename(p) in manual_filenames
]

print("Total imágenes:", len(all_image_paths))

if os.path.exists(QWEN_JSON_OUT):
    with open(QWEN_JSON_OUT, "r", encoding="utf-8") as f:
        results = json.load(f)
else:
    results = []

processed = set(r["filename"] for r in results)

pending_paths = [
    p for p in all_image_paths
    if os.path.basename(p) not in processed
]

print("Pendientes:", len(pending_paths))


def save_qwen_results(results):
    with open(QWEN_JSON_OUT, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    rows = []

    for r in results:
        row = {
            "filename": r["filename"],
            "image_path": r["image_path"],
            "model": r["model"]
        }

        for field in EVAL_QUESTIONS.keys():
            row[f"{field}_raw"] = r["raw_answers"].get(field)
            row[field] = r["predictions"].get(field)

        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(QWEN_CSV_OUT, index=False)


SAVE_EVERY = 10

for idx, path in enumerate(tqdm(pending_paths)):

    try:
        result = analyze_image_qwen(path)
        results.append(result)

    except Exception as e:
        print("\nERROR:", os.path.basename(path))
        print(e)

        results.append({
            "filename": os.path.basename(path),
            "image_path": path,
            "model": "qwen2_vl_2b",
            "error": str(e),
            "raw_answers": {},
            "predictions": {}
        })

    if (idx + 1) % SAVE_EVERY == 0:
        save_qwen_results(results)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"\nGuardadas {idx + 1} imágenes")

save_qwen_results(results)

print("\nProceso terminado")
print("CSV:", QWEN_CSV_OUT)

In [ ]:
manual_df = pd.read_csv(MANUAL_LABELS_PATH)
pred_df = pd.read_csv(QWEN_CSV_OUT)

eval_fields = [
    "has_tattoo",
    "tattoo_size",
    "tattoo_color",
    "contains_text",
    "tattoo_count"
]

merged = manual_df.merge(
    pred_df,
    on="filename",
    suffixes=("_manual", "_qwen")
)

print("Imágenes comparadas:", len(merged))

scores = []

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_qwen"

    correct = merged[manual_col] == merged[pred_col]

    scores.append({
        "model": "qwen2_vl_2b",
        "field": field,
        "correct": int(correct.sum()),
        "total": int(len(correct)),
        "accuracy": float(correct.mean())
    })

qwen_scores_df = pd.DataFrame(scores)
display(qwen_scores_df)

In [ ]:
for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_qwen"
    raw_col = field + "_raw"

    errors = merged[merged[manual_col] != merged[pred_col]][[
        "filename",
        manual_col,
        pred_col,
        raw_col
    ]]

    print("\n" + "=" * 80)
    print("Errores en:", field)
    print("=" * 80)
    display(errors)

ELIMINAR PARA VOLVER A EJECUTAR (LIMPIAR)

In [ ]:
import os

files_to_delete = [
    #"/content/drive/MyDrive/Tattoo-YOLO/test/moondream2_eval/moondream2_predictions.json",
    #"/content/drive/MyDrive/Tattoo-YOLO/test/moondream2_eval/moondream2_predictions.csv",

    #"/content/drive/MyDrive/Tattoo-YOLO/test/llava_eval/llava_predictions.json",
    #"/content/drive/MyDrive/Tattoo-YOLO/test/llava_eval/llava_predictions.csv",

    #"/content/drive/MyDrive/Tattoo-YOLO/test/qwen2vl_eval/qwen2vl_predictions.json",
    #"/content/drive/MyDrive/Tattoo-YOLO/test/qwen2vl_eval/qwen2vl_predictions.csv",
]

for path in files_to_delete:
    if os.path.exists(path):
        os.remove(path)
        print("Eliminado:", path)
    else:
        print("No existe:", path)

print("\nLimpieza terminada.")

results = []
processed = set()
pending_paths = []

print("Variables reiniciadas.")

EVALUACIÓN DE RESULTADOS

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from IPython.display import display

BASE = "/content/drive/MyDrive/Tattoo-YOLO"
SPLIT = "test"

IMAGE_DIR = os.path.join(BASE, SPLIT, "images")

MANUAL_LABELS_PATH = os.path.join(BASE, SPLIT, "manual_labels_test.csv")
QWEN_CSV_OUT = os.path.join(BASE, SPLIT, "qwen2vl_eval", "qwen2vl_predictions.csv")

ANALYSIS_OUT_DIR = os.path.join(BASE, SPLIT, "qwen2vl_error_analysis")
os.makedirs(ANALYSIS_OUT_DIR, exist_ok=True)

eval_fields = [
    "has_tattoo",
    "tattoo_size",
    "tattoo_color",
    "contains_text",
    "tattoo_count"
]

In [ ]:
# Cargar y unir resultados
manual_df = pd.read_csv(MANUAL_LABELS_PATH)
pred_df = pd.read_csv(QWEN_CSV_OUT)

merged = manual_df.merge(
    pred_df,
    on="filename",
    suffixes=("_manual", "_pred")
)

print("Imágenes comparadas:", len(merged))
display(merged.head())

In [ ]:
# Precisión por característica
accuracy_rows = []

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_pred"

    correct = merged[manual_col] == merged[pred_col]

    accuracy_rows.append({
        "field": field,
        "correct": int(correct.sum()),
        "total": len(correct),
        "accuracy": correct.mean(),
        "errors": int((~correct).sum())
    })

accuracy_df = pd.DataFrame(accuracy_rows)
display(accuracy_df)

# Gráfico
plt.figure(figsize=(8, 5))
plt.bar(accuracy_df["field"], accuracy_df["accuracy"])
plt.ylim(0, 1)
plt.ylabel("Accuracy")
plt.title("Qwen2-VL accuracy by feature")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

out_path = os.path.join(ANALYSIS_OUT_DIR, "accuracy_by_feature.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Guardado en:", out_path)

In [ ]:
# Número de errores

plt.figure(figsize=(8, 5))
plt.bar(accuracy_df["field"], accuracy_df["errors"])
plt.ylabel("Number of errors")
plt.title("Qwen2-VL errors by feature")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()

out_path = os.path.join(ANALYSIS_OUT_DIR, "errors_by_feature.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Guardado en:", out_path)

In [ ]:
# Matrices de confusión

def plot_confusion_matrix(df, field, save=True):
    manual_col = field + "_manual"
    pred_col = field + "_pred"

    labels = sorted(
        list(
            set(df[manual_col].dropna().unique())
            | set(df[pred_col].dropna().unique())
        )
    )

    cm = pd.crosstab(
        df[manual_col],
        df[pred_col],
        rownames=["Manual"],
        colnames=["Predicted"],
        dropna=False
    ).reindex(index=labels, columns=labels, fill_value=0)

    plt.figure(figsize=(6, 5))
    plt.imshow(cm.values)
    plt.title(f"Confusion matrix: {field}")
    plt.xlabel("Predicted")
    plt.ylabel("Manual")

    plt.xticks(range(len(cm.columns)), cm.columns, rotation=45, ha="right")
    plt.yticks(range(len(cm.index)), cm.index)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm.values[i, j], ha="center", va="center")

    plt.tight_layout()

    if save:
        out_path = os.path.join(ANALYSIS_OUT_DIR, f"confusion_{field}.png")
        plt.savefig(out_path, dpi=300)
        print("Guardado en:", out_path)

    plt.show()

    return cm

confusion_matrices = {}

for field in eval_fields:
    cm = plot_confusion_matrix(merged, field)
    confusion_matrices[field] = cm
    display(cm)

In [ ]:
# Falsos positivos y negativos
binary_fields = {
    "has_tattoo": ["yes", "no"],
    "tattoo_color": ["colored", "black_and_gray"],
    "contains_text": ["yes", "no"],
    "tattoo_count": ["multiple", "single"]
}

binary_error_rows = []

for field, positive_negative in binary_fields.items():
    positive, negative = positive_negative

    manual_col = field + "_manual"
    pred_col = field + "_pred"

    tp = ((merged[manual_col] == positive) & (merged[pred_col] == positive)).sum()
    tn = ((merged[manual_col] == negative) & (merged[pred_col] == negative)).sum()
    fp = ((merged[manual_col] == negative) & (merged[pred_col] == positive)).sum()
    fn = ((merged[manual_col] == positive) & (merged[pred_col] == negative)).sum()

    binary_error_rows.append({
        "field": field,
        "positive_label": positive,
        "negative_label": negative,
        "TP": int(tp),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn)
    })

binary_errors_df = pd.DataFrame(binary_error_rows)
display(binary_errors_df)

In [ ]:
# Grafico FP/FN

x = np.arange(len(binary_errors_df))
width = 0.35

plt.figure(figsize=(8, 5))
plt.bar(x - width/2, binary_errors_df["FP"], width, label="False positives")
plt.bar(x + width/2, binary_errors_df["FN"], width, label="False negatives")

plt.xticks(x, binary_errors_df["field"], rotation=30, ha="right")
plt.ylabel("Count")
plt.title("False positives and false negatives by feature")
plt.legend()
plt.tight_layout()

out_path = os.path.join(ANALYSIS_OUT_DIR, "false_positive_false_negative.png")
plt.savefig(out_path, dpi=300)
plt.show()

print("Guardado en:", out_path)

In [ ]:
# Errores concretos por característica

for field in eval_fields:
    manual_col = field + "_manual"
    pred_col = field + "_pred"
    raw_col = field + "_raw"

    errors = merged[merged[manual_col] != merged[pred_col]][[
        "filename",
        manual_col,
        pred_col,
        raw_col
    ]]

    print("\n" + "=" * 100)
    print("Errores en:", field)
    print("=" * 100)
    display(errors.head(30))

    errors.to_csv(
        os.path.join(ANALYSIS_OUT_DIR, f"errors_{field}.csv"),
        index=False
    )

In [ ]:
# Contar fallos por imagen

for field in eval_fields:
    merged[f"{field}_correct"] = (
        merged[f"{field}_manual"] == merged[f"{field}_pred"]
    )

correct_cols = [f"{field}_correct" for field in eval_fields]

merged["num_correct"] = merged[correct_cols].sum(axis=1)
merged["num_errors"] = len(eval_fields) - merged["num_correct"]

display(
    merged[[
        "filename",
        "num_correct",
        "num_errors"
    ] + [f"{field}_manual" for field in eval_fields] + [f"{field}_pred" for field in eval_fields]]
    .sort_values("num_errors", ascending=False)
    .head(20)
)

merged.to_csv(
    os.path.join(ANALYSIS_OUT_DIR, "merged_with_error_counts.csv"),
    index=False
)

In [ ]:
# Imágenes con todo bien

all_correct = merged[merged["num_errors"] == 0]

print("Imágenes con todo correcto:", len(all_correct))

display(all_correct[["filename", "num_correct", "num_errors"]].head(20))

In [ ]:
# Imágenes con más errores

most_errors = merged.sort_values("num_errors", ascending=False)

print("Casos con más errores:")
display(
    most_errors[[
        "filename",
        "num_correct",
        "num_errors"
    ] + [f"{field}_manual" for field in eval_fields] + [f"{field}_pred" for field in eval_fields]]
    .head(10)
)

In [ ]:
# Imágenes destacadas

def show_case(row, title=None):
    image_path = row["image_path_manual"] if "image_path_manual" in row else row["image_path"]

    img = Image.open(image_path).convert("RGB")

    if title:
        print(title)

    print("Filename:", row["filename"])
    display(img)

    data = []

    for field in eval_fields:
        data.append({
            "field": field,
            "manual": row[f"{field}_manual"],
            "predicted": row[f"{field}_pred"],
            "correct": row[f"{field}_manual"] == row[f"{field}_pred"]
        })

    display(pd.DataFrame(data))


# Mostrar algunos casos con todo correcto
print("=== EJEMPLOS CON TODO CORRECTO ===")

for _, row in all_correct.head(3).iterrows():
    show_case(row, "All correct example")


# Mostrar algunos casos con más errores
print("=== EJEMPLOS CON MÁS ERRORES ===")

for _, row in most_errors.head(3).iterrows():
    show_case(row, "High-error example")

In [ ]:
# Caso especial, no detecta tatuaje

has_tattoo_errors = merged[
    (merged["has_tattoo_manual"] == "yes") &
    (merged["has_tattoo_pred"] == "no")
]

print("Casos donde predice no tattoo pero manual es yes:", len(has_tattoo_errors))

display(has_tattoo_errors[[
    "filename",
    "has_tattoo_manual",
    "has_tattoo_pred",
    "has_tattoo_raw"
]])

for _, row in has_tattoo_errors.iterrows():
    show_case(row, "False negative: tattoo not detected")

In [ ]:
summary_path = os.path.join(ANALYSIS_OUT_DIR, "qwen2vl_summary_metrics.csv")
accuracy_df.to_csv(summary_path, index=False)

binary_path = os.path.join(ANALYSIS_OUT_DIR, "qwen2vl_binary_fp_fn.csv")
binary_errors_df.to_csv(binary_path, index=False)

print("Resumen guardado en:", summary_path)
print("FP/FN guardado en:", binary_path)

In [ ]:
# Errores relevantes de tamaño

size_errors = merged[
    merged["tattoo_size_manual"] != merged["tattoo_size_pred"]
].copy()

severe_size_errors = merged[
    (
        (merged["tattoo_size_manual"] == "small") &
        (merged["tattoo_size_pred"] == "large")
    )
    |
    (
        (merged["tattoo_size_manual"] == "large") &
        (merged["tattoo_size_pred"] == "small")
    )
].copy()

print("Total errores en tattoo_size:", len(size_errors))
print("Errores graves small <-> large:", len(severe_size_errors))

print("\nPorcentaje de errores graves sobre todos los errores de tamaño:")
print(len(severe_size_errors) / len(size_errors) if len(size_errors) > 0 else 0)

print("\nPorcentaje de errores graves sobre todo el test:")
print(len(severe_size_errors) / len(merged) if len(merged) > 0 else 0)

display(severe_size_errors[[
    "filename",
    "tattoo_size_manual",
    "tattoo_size_pred",
    "tattoo_size_raw"
]])

In [ ]:
size_cm = pd.crosstab(
    merged["tattoo_size_manual"],
    merged["tattoo_size_pred"],
    rownames=["Manual"],
    colnames=["Predicted"],
    dropna=False
).reindex(
    index=["small", "medium", "large"],
    columns=["small", "medium", "large"],
    fill_value=0
)

display(size_cm)

severe_small_to_large = size_cm.loc["small", "large"]
severe_large_to_small = size_cm.loc["large", "small"]

print("Small manual predicho como large:", severe_small_to_large)
print("Large manual predicho como small:", severe_large_to_small)
print("Total errores graves:", severe_small_to_large + severe_large_to_small)

In [ ]:
# ============================================
# Métrica tolerante para tattoo_size
# Solo penaliza errores extremos small <-> large
# ============================================

def is_size_severe_error(row):
    manual = row["tattoo_size_manual"]
    pred = row["tattoo_size_pred"]

    return (
        (manual == "small" and pred == "large") or
        (manual == "large" and pred == "small")
    )

merged["tattoo_size_severe_error"] = merged.apply(is_size_severe_error, axis=1)

severe_error_count = int(merged["tattoo_size_severe_error"].sum())
non_severe_count = len(merged) - severe_error_count

tolerant_accuracy = non_severe_count / len(merged)

print("Errores extremos small <-> large:", severe_error_count)
print("Total imágenes:", len(merged))
print("Tattoo size tolerant accuracy:", tolerant_accuracy)

In [ ]:
print("Ejemplos de errores graves en tattoo_size:")

for _, row in severe_size_errors.head(10).iterrows():
    show_case(row, "Severe tattoo_size error")